# Imports & Configuration

In [0]:
import warnings
warnings.filterwarnings('ignore')

In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

In [0]:
sc = spark.sparkContext

# Reading File


In [0]:
transactions_file = '/Volumes/spark_data/dev/spark_query_plans/data/data_skew/transactions.parquet/'

df_trans = spark.read.parquet(transactions_file)

In [0]:
display(df_trans)

In [0]:
customers_file ="/Volumes/spark_data/dev/spark_query_plans/data/data_skew/customers.parquet/"
df_customers = spark.read.parquet(customers_file)


In [0]:
display(df_customers)

# Narrow Transformations
1. filter rows where city='boston'
2. add a new column adding first_name and last_name spliting name column
3. alter an existing column adding 5 to age column
4. select relevant columns

In [0]:
df_narrow_transform = (
    df_customers
        .filter(F.col('city') =='boston')
        .withColumn('first_name',F.split('name',' ').getItem(0))
        .withColumn('last_name', F.split('name',' ').getItem(1))
        .withColumn('age', F.col('age')+ F.lit(5))
        .select("cust_id","first_name","last_name","age","gender","birthday")
)
display(df_narrow_transform)
df_narrow_transform.explain(True)

# Wide Transformations
1. Repartition
2. Coalesce
3. Joins
4. GroupBy
    1. count
    2. countDistinct
    3. sum

1. Repartition 
distributing your dataset 

In [0]:
spark.conf.set('spark.sql.adaptive.enabled', True)

In [0]:
df_trans.rdd.getNumPartitions()

In [0]:
df_trans.repartition(24).explain(True)

# Coalesce

In [0]:
df_trans.rdd.getNumPartitions()

In [0]:
df_trans.coalesce(1).explain(True)

# Joins

In [0]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold',-1)

In [0]:
df_joined =(
    df_trans.join(df_customers,
                  how ='inner',
                  on ='cust_id'
                  )
)

In [0]:
df_joined.explain(True)

# Group By

In [0]:
df_trans.printSchema()

In [0]:
df_city_counts = (
    df_trans
    .groupBy('city')
    .count()
)

In [0]:
df_city_counts.explain(True)

In [0]:
df_txn_amt_city = (
    df_trans
    .groupBy("city")
    .agg(F.sum("amt").alias('txn_amount'))
)

In [0]:
df_txn_amt_city.explain(True)

# Group By Distinct Count

In [0]:
df_txn_per_city = (
    df_trans
    .groupBy("cust_id")
    .agg(F.countDistinct("cust_id").alias("city_count"))
)

In [0]:
df_txn_per_city.explain(True)
